# TP 1

In [64]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np

In [2]:
food_parquet = '../../data/food.parquet'
parquet_file = pq.ParquetFile(food_parquet)

In [10]:
parquet_columns = ["code", "brands", "product_name", "nutriments", "categories_tags", "countries_tags", "nutriscore_grade"]
first_batch = next(parquet_file.iter_batches(100_000, columns=parquet_columns))
df = first_batch.to_pandas()
df.to_parquet("../../data/food_light.parquet")

In [11]:
df = pd.read_parquet("../../data/food_light.parquet")

df_france = df[df['countries_tags'].astype(str).str.lower().str.contains('france', na=False)]

df_france

,code,brands,product_name,nutriments,categories_tags,countries_tags,nutriscore_grade
0,0000101209159,Bovetti,"[{'lang': 'main', 'text': 'Véritable pâte à ta...","[{'name': 'fruits-vegetables-nuts', 'value': N...","[en:breakfasts, en:spreads, en:sweet-spreads, ...",[en:france],e
17,0000130008136,NaN,"[{'lang': 'main', 'text': 'Escalope de dinde'}...",None,"[en:meats-and-their-products, en:meats, en:pou...",[en:france],unknown
18,0000140323687,NaN,"[{'lang': 'main', 'text': 'Madeleine Framboise...",None,"[en:snacks, en:sweet-snacks, en:biscuits-and-c...",[en:france],unknown
19,0000141013129,,"[{'lang': 'main', 'text': 'Croissants margarin...",None,"[en:snacks, en:sweet-snacks, en:sweet-pastries...",[en:france],unknown
24,0000171812457,NaN,"[{'lang': 'main', 'text': 'Glaces vegetales de...","[{'name': 'energy-kcal', 'value': None, '100g'...",None,[en:france],unknown
...,...,...,...,...,...,...,...
99761,0077782030181,Johnsonville,"[{'lang': 'main', 'text': 'Beddar with Cheddar...","[{'name': 'iron', 'value': None, '100g': 0.000...","[en:meats-and-their-products, en:meats, en:pre...","[en:france, en:united-states]",e
99842,0077885710614,Tapatío,"[{'lang': 'main', 'text': 'Hot Sauce'}, {'lang...","[{'name': 'added-sugars', 'value': None, '100g...","[en:condiments, en:sauces, en:hot-sauces, en:G...","[en:france, en:germany, en:united-states]",e
99844,0077885892020,Tapatío,"[{'lang': 'main', 'text': 'Hot Sauce'}, {'lang...","[{'name': 'sodium', 'value': None, '100g': 1.7...","[en:condiments, en:sauces, en:hot-sauces, en:g...","[en:france, en:united-states]",e
99892,0077890269640,Wegmans,"[{'lang': 'main', 'text': 'Frizzante sicilian ...","[{'name': 'carbohydrates', 'value': None, '100...",None,[en:france],unknown


In [ ]:
# Combien de produits sont vendus en France ?
print(f"Nombre de produits vendus en France : {len(df_france)}")

Nombre de produits vendus en France : 6320


In [ ]:
# Quelle part du catalogue possède un Nutri-Score renseigné ?
nutriscore_grade = ['a', 'b', 'c', 'd', 'e']
count = 0
for nutriscore in df_france['nutriscore_grade']:
    if nutriscore.lower() in nutriscore_grade :
        count += 1
part_nutriscore = round((100 * count)/len(df_france), 2)
print(f"Part des Nutriscore renseigné : {part_nutriscore}%")

Part des Nutriscore renseigné : 59.3%


In [ ]:
# Quelles sont les dix marques les plus présentes ?
brands = {}
for brand in df_france['brands'] :
    brand_str = str(brand)
    if brand_str.lower() not in brands:
        brands[brand_str.lower()] = 1
    else:
        brands[brand_str.lower()] += 1 

brands_sorted = dict(sorted(brands.items(), key=lambda item: item[1], reverse=True))
top_10_brands = dict(list(brands_sorted.items())[:10])
print(top_10_brands)

{'nan': 782, 'marks & spencer': 635, 'm&s': 215, '': 120, "sainsbury's": 113, 'marks and spencer': 75, 'm&s food': 68, "kellogg's": 60, 'twinings': 58, "reese's": 56}


In [67]:
# Quel est le taux de valeurs manquantes sur les nutriments clés (energy_100g, sugars_100g, salt_100g) ?

nutriments = round(df_france['nutriments'].isna().mean()*100, 2)
print(f"Part des nutriments non remplis : {nutriments}%")

def is_missing(list_nutriments, name_nutriment):
    # Si la case entière est vide (NaN, None, float au lieu d'une liste...)
    if not isinstance(list_nutriments, (list, np.ndarray)):
        return True
        
    # On parcourt chaque dictionnaire de la liste
    for nutriment in list_nutriments:
        # Si on trouve le bon nutriment (ex: 'energy')
        if isinstance(nutriment, dict) and nutriment.get('name') == name_nutriment:
            valeur = nutriment.get('100g')
            # On vérifie si sa valeur est vide (None, NaN, ou chaîne vide)
            if valeur is None or pd.isna(valeur) or valeur == '':
                return True
            else:
                return False # On a trouvé une valeur valide !
                
    # Si on a fini la liste et qu'on n'a pas trouvé le nutriment, il est manquant
    return True

total_lines = len(df_france)

# On applique la fonction pour chaque nutriment ciblé
# (le .sum() va additionner tous les True, donc compter les manquants)
energy_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'energy')).sum()
sugars_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'sugars')).sum()
salt_missing = df_france['nutriments'].apply(lambda x: is_missing(x, 'salt')).sum()

# On calcule le taux (en pourcentage)
taux_energy = (energy_missing / total_lines) * 100
taux_sugars = (sugars_missing / total_lines) * 100
taux_salt = (salt_missing / total_lines) * 100

# Affichage propre des résultats
print("Taux de valeurs manquantes :")
print(f"- Énergie : {taux_energy:.2f} % ({energy_missing} manquants sur {total_lines})")
print(f"- Sucres  : {taux_sugars:.2f} % ({sugars_missing} manquants sur {total_lines})")
print(f"- Sel     : {taux_salt:.2f} % ({salt_missing} manquants sur {total_lines})")

Part des nutriments non remplis : 5.71%
Taux de valeurs manquantes :
- Énergie : 7.88 % (498 manquants sur 6320)
- Sucres  : 9.10 % (575 manquants sur 6320)
- Sel     : 11.44 % (723 manquants sur 6320)


In [ ]:
# Selon toi, qu'est-ce qui semble le plus « sale » ou atypique dans ces données ?
# La colonne 'brands', car ses données sont soit NaN, soit vide, soit se répète, un léger changement d'orthographe qui par conséquent se considère comme unique. 

# TP 2